# SRQ generalization M9 — repeated whole-process systems audit

Train-only CIFAR-100 measurement with four paired repetitions and eight isolated workers. Method order is balanced. Feature extraction is intentionally repeated; no test split is opened.

In [ ]:
# Fresh repository and dependencies.
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
clone=subprocess.run(['git','clone','--depth','1','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],text=True,capture_output=True)
if clone.returncode!=0:
    print('CLONE STDOUT:\n',clone.stdout); print('CLONE STDERR:\n',clone.stderr)
    raise RuntimeError('Repository clone failed; inspect CLONE STDERR above.')
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','nvidia-ml-py'],check=True)
print('REPO COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

In [ ]:
# Locked source identities. This cell must pass before expensive work.
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG='configs/srq_generalization_m9_repeated_systems_train_only.json'
RUNNER='tools/srq_generalization_m9.py'
EXPECTED={
 'configs/srq_generalization_m9_repeated_systems_train_only.json':'8d49d927e9adf532f1c2536fef1a2ab6b24f04febdd3da10c1ed736c648a0787',
 'tools/srq_generalization_m9.py':'94e190cf664f716b7d5952044a2782a1548acd26b6d104c81b134a08aef48482',
 'configs/srq_fly_priority5_cifar100_whole_process_memory.json':'ba02e0e742fdaf17e364d6ef182d8e32f8220ac5140253ec2155572054db75da',
 'tools/srq_fly_priority5_memory.py':'7ae9397d3e26d8eeec03ad13b76adcbb3f778797d6d587cf64b4f8c8fdea2c94',
 'methods/srq_fly_optimized/learner.py':'40edac2e2cc88faac549f5c87217f3143d815bf53ecad8a37dfdb22c112691ae',
 'methods/srq_fly_optimized/storage.py':'9d288a3661985da657371e8581f406825d4a8d5e6e0c63381aacda8484490986',
 'tools/srq_fly_system_benchmark.py':'85e2d7f8a27f8081148a88bf88dcd374af20aa39f2a77691c9ca744a2ff0d96c',
 'models/backbone.py':'90f70c9a2b16e4435e6e348ba701083a17de830d9a3d9ba080695e23d333f58b',
 'models/flyhash.py':'24ba321a71f735031b0da430ab4d3519e54e6c3149fc6c913c63b4172f6712cb',
 'utils/data_utils.py':'cad262c013dbbd85c6bcd790b9276882ba1ff0a15b2bae57bff5a20293e9e5d8',
 'utils/train_utils.py':'e24983bd3042ad82ec069916ba2853cf1c818cb2911ce193710c8ccd70e86bda'}
for path,expected in EXPECTED.items(): assert sha(path)==expected,(path,sha(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
subprocess.run([sys.executable,'-m','pytest','-q','-p','no:cacheprovider','tests/test_srq_generalization_m9.py','tests/test_srq_fly_priority5_memory.py'],check=True)
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True).strip())
print('M9 LOCKED SOURCE AND UNIT GATES: PASS')

In [ ]:
# Download public training data and the locked frozen checkpoint.
import kagglehub
from huggingface_hub import hf_hub_download
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
CHECKPOINT_PATH=hf_hub_download('timm/vit_base_patch16_224.augreg2_in21k_ft_in1k','model.safetensors')
assert Path(CIFAR_ROOT).exists()
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
from tools.srq_fly_priority5_memory import _NVMLSampler, _build_train_loader, _read_config
P5_CONFIG=Path('configs/srq_fly_priority5_cifar100_whole_process_memory.json')
preflight_root=Path('/content/srq_m9_loader_preflight')
loader=_build_train_loader(_read_config(P5_CONFIG),Path(CIFAR_ROOT),preflight_root)
images,labels=next(iter(loader))
assert tuple(images.shape[1:])==(3,224,224) and images.dtype.is_floating_point
assert labels.ndim==1 and len(images)==len(labels)
del images,labels,loader
shutil.rmtree(preflight_root)
sampler=_NVMLSampler(0)
device_bytes,parent_bytes=sampler.sample(os.getpid())
print('NVML PREFLIGHT:',sampler.device_name,'device MiB=',device_bytes/2**20,'parent process bytes=',parent_bytes)
sampler.close()
print('DATA, CHECKPOINT, LOADER, AND NVML PREFLIGHT: PASS')

In [ ]:
# Long cell: four paired repetitions, eight isolated whole-process workers.
OUTPUT_DIR='/content/srq_m9_repeated_systems'
SCRATCH_DIR='/content/srq_m9_scratch'
for path in (OUTPUT_DIR,SCRATCH_DIR):
    if Path(path).exists(): shutil.rmtree(path)
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--output-dir',OUTPUT_DIR,'--scratch-dir',SCRATCH_DIR,'--device','cuda','--require-clean-git']
print('M9 START: 4 paired repetitions, 8 isolated workers; expect roughly 35-50 minutes on T4.',flush=True)
LOG_PATH='/content/srq_m9_runner.log'
with open(LOG_PATH,'w',encoding='utf-8') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line); log.flush()
    return_code=process.wait()
print('M9 RETURN CODE:',return_code,'| log:',LOG_PATH)
RESULT=Path(OUTPUT_DIR)/'m9_results.json'
assert RESULT.is_file(),'M9 failed before writing results; return the complete live output.'
payload=json.loads(RESULT.read_text())
print('M9 STATUS:',payload['status'])
print('M9 GATES:',json.dumps(payload['gates'],indent=2))
assert return_code==0 and payload['status']=='PASS_M9_REPEATED_SYSTEMS_TRAIN_ONLY','M9 failed; preserve all outputs and do not relax gates.'

In [ ]:
# Compact paper-facing aggregate. Values are mean +/- sample SD over n=4.
import pandas as pd
metrics=[
 ('persistent_state_bytes','Persistent state','MiB',2**20),
 ('serialized_checkpoint_bytes','Checkpoint','MiB',2**20),
 ('torch_analytic_peak_allocated_bytes','PyTorch analytic peak','MiB',2**20),
 ('nvml_whole_process_peak_process_bytes','NVML process peak','MiB',2**20),
 ('analytic_stage_seconds','Analytic stage','s',1),
 ('total_measured_stage_seconds','Total measured stages','s',1)]
rows=[]
for method in ('exact_fly_10000','srq_fly_p2b_10000'):
    for key,label,unit,scale in metrics:
        value=payload['aggregate'][method][key]
        rows.append({'method':method,'metric':label,'mean':value['mean']/scale,'sample_std':value['sample_std']/scale,'unit':unit})
display(pd.DataFrame(rows))
print('PAIRED SRQ/EXACT RATIOS:')
print(json.dumps(payload['paired_srq_over_exact_ratios'],indent=2))

In [ ]:
# Inspect runner-generated vector figures.
from IPython.display import SVG, display
for name in ('m9_memory_summary.svg','m9_stage_time_summary.svg'):
    path=Path(OUTPUT_DIR)/name
    assert path.is_file()
    display(SVG(filename=str(path)))

In [ ]:
# Export evidence only; exclude dataset, checkpoint, scratch views/features, and temporary checkpoints.
BUNDLE=Path('/content/srq_generalization_m9_repeated_systems_train_only')
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
shutil.copytree(OUTPUT_DIR,BUNDLE/'results')
shutil.copy2(CONFIG,BUNDLE/'config.json')
shutil.copy2('docs/research/SRQ_GENERALIZATION_M9_RUNBOOK.md',BUNDLE/'runbook.md')
manifest={'artifact':'srq_generalization_m9_repeated_systems_train_only','uses_test_set':False,'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'source_hashes':EXPECTED,'result_sha256':sha(RESULT)}
(BUNDLE/'manifest.json').write_text(json.dumps(manifest,indent=2))
ARCHIVE=Path(shutil.make_archive(str(BUNDLE),'zip',root_dir=BUNDLE.parent,base_dir=BUNDLE.name))
with zipfile.ZipFile(ARCHIVE) as archive: assert archive.testzip() is None
print('ZIP:',ARCHIVE,'bytes=',ARCHIVE.stat().st_size,'sha256=',sha(ARCHIVE))
from google.colab import files
files.download(str(ARCHIVE))